# 04 — ML Stage: GNN Training for Interpretability Prediction

Train Graph Neural Networks to predict interpretability scores from
attribution graph structure. We compare:
1. **StructuralMLP** — baseline MLP on hand-crafted structural metrics
2. **AttributionGNN (GAT)** — learns directly from graph topology
3. **Hybrid GNN** — GNN + structural feature fusion


In [27]:
import sys
sys.path.insert(0, "..")

import importlib
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import src.graph_generator
import src.structural_metrics
import src.dataset
import src.model
import src.train
importlib.reload(src.graph_generator)
importlib.reload(src.structural_metrics)
importlib.reload(src.dataset)
importlib.reload(src.model)
importlib.reload(src.train)

from src.graph_generator import AttributionGraph
from src.structural_metrics import compute_all_metrics, StructuralMetrics
from src.dataset import load_graphs_from_dir, attribution_graph_to_pyg, create_splits
from src.model import StructuralMLP, AttributionGNN
from src.train import Trainer, TrainingConfig, r2_score
from src.utils import plot_training_history

try:
    from torch_geometric.loader import DataLoader as PyGDataLoader
except ImportError:
    from torch_geometric.data import DataLoader as PyGDataLoader

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print("Imports loaded.")


Using device: mps
Imports loaded.


## 1. Prepare Dataset

Convert all attribution graphs to PyG Data objects with node features,
edge connectivity, and structural metrics.


In [28]:
synthetic_graphs = load_graphs_from_dir("../data/raw/synthetic")

real_graphs = []
for cat in ["factual_recall", "reasoning", "creative_writing",
            "code_understanding", "ambiguous_context", "multilingual"]:
    from pathlib import Path
    cat_dir = Path(f"../data/raw/{cat}")
    if cat_dir.exists():
        cat_graphs = load_graphs_from_dir(str(cat_dir))
        for g in cat_graphs:
            g.metadata["category"] = cat
        real_graphs.extend(cat_graphs)

all_graphs = synthetic_graphs + real_graphs
print(f"Total graphs: {len(all_graphs)}")


Loaded 375 graphs from ../data/raw/synthetic
Loaded 5 graphs from ../data/raw/factual_recall
Loaded 5 graphs from ../data/raw/reasoning
Loaded 0 graphs from ../data/raw/creative_writing
Loaded 0 graphs from ../data/raw/code_understanding
Loaded 0 graphs from ../data/raw/ambiguous_context
Loaded 0 graphs from ../data/raw/multilingual
Total graphs: 385


In [29]:
data_list = []
for i, g in enumerate(all_graphs):
    try:
        data = attribution_graph_to_pyg(g, graph_id=f"graph_{i:04d}")
        if data is not None and data.num_nodes > 0 and data.y is not None:
            data_list.append(data)
    except Exception as e:
        pass    # skip problematic graphs
    if (i + 1) % 100 == 0:
        print(f"  Converted {i+1}/{len(all_graphs)}")

print(f"\nValid PyG graphs: {len(data_list)}")
print(f"Node feature dim: {data_list[0].x.shape[1]}")
print(f"Structural feature dim: {data_list[0].structural_features.shape[0]}")

# Label distribution
labels = [d.y.item() for d in data_list]
from collections import Counter
print(f"Label distribution: {Counter(labels)}")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:302: RuntimeWarning: invalid value encountered in scalar divide
  return float((xy * (M - ab)).sum() / np.sqrt(vara * varb))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:302: RuntimeWarning: invalid value encountered in sqrt
  return float((xy * (M - ab)).sum() / np.sqrt(vara * varb))


  Converted 100/385
  Converted 200/385
  Converted 300/385

Valid PyG graphs: 375
Node feature dim: 7
Structural feature dim: 34
Label distribution: Counter({1.0: 150, 0.0: 150, 0.5: 75})


In [30]:
# Split train/val/test
torch.manual_seed(42)
np.random.seed(42)

n = len(data_list)
indices = np.random.permutation(n)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_data = [data_list[i] for i in indices[:n_train]]
val_data = [data_list[i] for i in indices[n_train:n_train + n_val]]
test_data = [data_list[i] for i in indices[n_train + n_val:]]

batch_size = 32
train_loader = PyGDataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = PyGDataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = PyGDataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")


Train: 262, Val: 56, Test: 57


## 2. Baseline: Structural MLP

First, train a simple MLP on the hand-crafted structural metrics.
This tells us how much signal is in the metrics we designed.


In [31]:
structural_dim = data_list[0].structural_features.shape[0]

mlp_config = TrainingConfig(
    model_type="mlp",
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=100,
    patience=15,
    batch_size=32,
    log_dir="../results/metrics/mlp",
)

mlp_trainer = Trainer(mlp_config)
mlp_model = mlp_trainer.create_model(structural_feature_dim=structural_dim)
print(f"MLP parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

mlp_model = mlp_trainer.train(mlp_model, train_loader, val_loader)


MLP parameters: 15,297
Training on mps
Model: mlp, Epochs: 100
Train size: 262, Val size: 56
----------------------------------------------------------------------
Epoch   1 | Train Loss: 1.2107 | Val Loss: 0.7722 | Val MAE: 0.8671 | Val R2: -2.9567 | LR: 0.001000 | 0.1s
Epoch   5 | Train Loss: 0.2830 | Val Loss: 0.1438 | Val MAE: 0.3751 | Val R2: 0.2632 | LR: 0.001000 | 0.1s
Epoch  10 | Train Loss: 0.1314 | Val Loss: 0.0261 | Val MAE: 0.1459 | Val R2: 0.8660 | LR: 0.001000 | 0.1s
Epoch  15 | Train Loss: 0.0780 | Val Loss: 0.0032 | Val MAE: 0.0509 | Val R2: 0.9837 | LR: 0.001000 | 0.1s
Epoch  20 | Train Loss: 0.0467 | Val Loss: 0.0035 | Val MAE: 0.0458 | Val R2: 0.9823 | LR: 0.001000 | 0.1s
Epoch  25 | Train Loss: 0.0459 | Val Loss: 0.0011 | Val MAE: 0.0263 | Val R2: 0.9945 | LR: 0.000500 | 0.1s
Epoch  30 | Train Loss: 0.0359 | Val Loss: 0.0052 | Val MAE: 0.0604 | Val R2: 0.9731 | LR: 0.000500 | 0.1s
Epoch  35 | Train Loss: 0.0420 | Val Loss: 0.0035 | Val MAE: 0.0506 | Val R2: 0.9822 |

In [32]:
criterion = nn.MSELoss()
mlp_test = mlp_trainer.evaluate(mlp_model, test_loader, criterion)
print(f"MLP Test Results:")
print(f"  Loss: {mlp_test["loss"]:.4f}")
print(f"  MAE:  {mlp_test["mae"]:.4f}")
print(f"  R2:   {mlp_test["r2"]:.4f}")


MLP Test Results:
  Loss: 0.0003
  MAE:  0.0156
  R2:   0.9982


## 3. Graph Neural Network (GAT)

Now train a Graph Attention Network that learns directly from the
attribution graph topology. The GNN should capture patterns that
our hand-crafted metrics might miss.


In [33]:
node_dim = data_list[0].x.shape[1]

gnn_config = TrainingConfig(
    model_type="gnn",
    gnn_type="GAT",
    hidden_dim=64,
    num_gnn_layers=3,
    num_heads=4,
    use_structural_fusion=False,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=100,
    patience=15,
    batch_size=32,
    log_dir="../results/metrics/gnn",
)

gnn_trainer = Trainer(gnn_config)
gnn_model = gnn_trainer.create_model(node_feature_dim=node_dim)
print(f"GNN parameters: {sum(p.numel() for p in gnn_model.parameters()):,}")

gnn_model = gnn_trainer.train(gnn_model, train_loader, val_loader)


GNN parameters: 20,097
Training on mps
Model: gnn, Epochs: 100
Train size: 262, Val size: 56
----------------------------------------------------------------------
Epoch   1 | Train Loss: 0.1837 | Val Loss: 0.3647 | Val MAE: 0.4561 | Val R2: -0.8687 | LR: 0.001000 | 0.3s
Epoch   5 | Train Loss: 0.0528 | Val Loss: 0.3917 | Val MAE: 0.4536 | Val R2: -1.0070 | LR: 0.001000 | 0.2s
Epoch  10 | Train Loss: 0.0366 | Val Loss: 0.3365 | Val MAE: 0.4282 | Val R2: -0.7242 | LR: 0.001000 | 0.3s
Epoch  15 | Train Loss: 0.0432 | Val Loss: 0.3898 | Val MAE: 0.4675 | Val R2: -0.9973 | LR: 0.000500 | 0.2s
Epoch  20 | Train Loss: 0.0317 | Val Loss: 0.3323 | Val MAE: 0.4187 | Val R2: -0.7026 | LR: 0.000500 | 0.2s
Epoch  25 | Train Loss: 0.0238 | Val Loss: 0.3564 | Val MAE: 0.4519 | Val R2: -0.8264 | LR: 0.000500 | 0.3s
Epoch  30 | Train Loss: 0.0235 | Val Loss: 0.2974 | Val MAE: 0.4094 | Val R2: -0.5238 | LR: 0.000250 | 0.3s
Epoch  35 | Train Loss: 0.0203 | Val Loss: 0.2987 | Val MAE: 0.4182 | Val R2: -0

In [34]:
gnn_test = gnn_trainer.evaluate(gnn_model, test_loader, criterion)
print(f"\nGNN (GAT) Test Results:")
print(f"  Loss: {gnn_test["loss"]:.4f}")
print(f"  MAE:  {gnn_test["mae"]:.4f}")
print(f"  R2:   {gnn_test["r2"]:.4f}")



GNN (GAT) Test Results:
  Loss: 0.2908
  MAE:  0.4184
  R2:   -0.5811


## 4. Hybrid Model: GNN + Structural Feature Fusion

Combine the GNN's learned representations with our hand-crafted
structural metrics. This tests whether both sources of information
are complementary.


In [35]:
hybrid_config = TrainingConfig(
    model_type="gnn",
    gnn_type="GAT",
    hidden_dim=64,
    num_gnn_layers=3,
    num_heads=4,
    use_structural_fusion=True,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=100,
    patience=15,
    batch_size=32,
    log_dir="../results/metrics/hybrid",
)

hybrid_trainer = Trainer(hybrid_config)
hybrid_model = hybrid_trainer.create_model(
    node_feature_dim=node_dim,
    structural_feature_dim=structural_dim,
)
print(f"Hybrid parameters: {sum(p.numel() for p in hybrid_model.parameters()):,}")

hybrid_model = hybrid_trainer.train(hybrid_model, train_loader, val_loader)


Hybrid parameters: 26,433
Training on mps
Model: gnn, Epochs: 100
Train size: 262, Val size: 56
----------------------------------------------------------------------
Epoch   1 | Train Loss: 0.2476 | Val Loss: 0.1376 | Val MAE: 0.2899 | Val R2: 0.2949 | LR: 0.001000 | 0.3s
Epoch   5 | Train Loss: 0.0444 | Val Loss: 0.0504 | Val MAE: 0.1865 | Val R2: 0.7416 | LR: 0.001000 | 0.3s
Epoch  10 | Train Loss: 0.0417 | Val Loss: 0.0450 | Val MAE: 0.1824 | Val R2: 0.7696 | LR: 0.000500 | 0.3s
Epoch  15 | Train Loss: 0.0373 | Val Loss: 0.0479 | Val MAE: 0.1879 | Val R2: 0.7547 | LR: 0.000250 | 0.3s
Epoch  20 | Train Loss: 0.0324 | Val Loss: 0.0391 | Val MAE: 0.1736 | Val R2: 0.7996 | LR: 0.000250 | 0.3s
Epoch  25 | Train Loss: 0.0267 | Val Loss: 0.0387 | Val MAE: 0.1707 | Val R2: 0.8017 | LR: 0.000250 | 0.2s
Epoch  30 | Train Loss: 0.0327 | Val Loss: 0.0448 | Val MAE: 0.1894 | Val R2: 0.7706 | LR: 0.000250 | 0.3s
Epoch  35 | Train Loss: 0.0294 | Val Loss: 0.0418 | Val MAE: 0.1823 | Val R2: 0.7860

In [36]:
hybrid_test = hybrid_trainer.evaluate(hybrid_model, test_loader, criterion)
print(f"\nHybrid (GNN + Structural) Test Results:")
print(f"  Loss: {hybrid_test["loss"]:.4f}")
print(f"  MAE:  {hybrid_test["mae"]:.4f}")
print(f"  R2:   {hybrid_test["r2"]:.4f}")



Hybrid (GNN + Structural) Test Results:
  Loss: 0.0441
  MAE:  0.1827
  R2:   0.7599


## 5. Model Comparison & Analysis


In [37]:
results = pd.DataFrame({
    "Model": ["Structural MLP", "GNN (GAT)", "Hybrid (GNN + Structural)"],
    "Test Loss (MSE)": [mlp_test["loss"], gnn_test["loss"], hybrid_test["loss"]],
    "Test MAE": [mlp_test["mae"], gnn_test["mae"], hybrid_test["mae"]],
    "Test R2": [mlp_test["r2"], gnn_test["r2"], hybrid_test["r2"]],
})
print("Model Comparison:")
print("=" * 65)
print(results.to_string(index=False))

results.to_csv("../results/metrics/model_comparison.csv", index=False)


Model Comparison:
                    Model  Test Loss (MSE)  Test MAE   Test R2
           Structural MLP         0.000332  0.015577  0.998197
                GNN (GAT)         0.290772  0.418363 -0.581118
Hybrid (GNN + Structural)         0.044146  0.182737  0.759948


In [38]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics_to_plot = [("Test Loss (MSE)", "lower is better"), 
                   ("Test MAE", "lower is better"),
                   ("Test R2", "higher is better")]

colors = ["#3498db", "#e74c3c", "#2ecc71"]

for ax, (metric, note) in zip(axes, metrics_to_plot):
    bars = ax.bar(results["Model"], results[metric], color=colors, alpha=0.8)
    ax.set_title(f"{metric}\n({note})")
    ax.set_ylabel(metric)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Model Performance Comparison", fontsize=14)
plt.tight_layout()
plt.savefig("../results/figures/model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved model_comparison.png")


Saved model_comparison.png


In [39]:
@torch.no_grad()
def get_predictions(model, loader, model_type="gnn", use_structural=False):
    model.eval()
    preds, actuals = [], []
    for batch in loader:
        batch = batch.to(device)
        sf = batch.structural_features
        if sf.dim() == 1:
            sf = sf.view(batch.num_graphs, -1)
        if model_type == "mlp":
            p = model(sf)
        else:
            p = model(batch.x, batch.edge_index, batch.batch,
                      edge_attr=batch.edge_attr,
                      structural_features=sf if use_structural else None)
        preds.extend(p.cpu().numpy().flatten())
        actuals.extend(batch.y.cpu().numpy().flatten())
    return np.array(preds), np.array(actuals)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_info = [
    (mlp_model, "Structural MLP", "mlp", False),
    (gnn_model, "GNN (GAT)", "gnn", False),
    (hybrid_model, "Hybrid", "gnn", True),
]

for ax, (model, name, mtype, use_struct) in zip(axes, models_info):
    preds, actuals = get_predictions(model, test_loader, mtype, use_struct)
    ax.scatter(actuals, preds, alpha=0.5, s=30, c="#3498db")
    ax.plot([0, 1], [0, 1], "r--", alpha=0.7, label="Perfect")
    r2 = r2_score(actuals, preds)
    title_str = name + " (R2=" + f"{r2:.3f}" + ")"
    ax.set_title(title_str)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.legend()
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)

plt.suptitle("Predicted vs Actual Interpretability Score", fontsize=14)
plt.tight_layout()
plt.savefig("../results/figures/prediction_scatter.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved prediction_scatter.png")


Saved prediction_scatter.png


In [40]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (trainer, name) in zip(axes, [(mlp_trainer, "MLP"), (gnn_trainer, "GNN"), (hybrid_trainer, "Hybrid")]):
    epochs = [h.epoch for h in trainer.history]
    ax.plot(epochs, [h.train_loss for h in trainer.history], label="Train", alpha=0.8)
    ax.plot(epochs, [h.val_loss for h in trainer.history], label="Val", alpha=0.8)
    ax.set_title(f"{name} Training Curve")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()

plt.suptitle("Training Curves", fontsize=14)
plt.tight_layout()
plt.savefig("../results/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved training_curves.png")


Saved training_curves.png
